In [ ]:
#I heavily recommend running this code in a Jupyter Notebook or similar environment to make package management easier
#This script needs over 50 packages to run, so it is not recommended to run it in a standard Python environment without proper package management.
#
#This script loads customer provided Excel data and shapefile data from Digiroad.
#These files are merged based on the road number and segment number.
#Condition is matched based on the AET and LET values from the shapefile and the Aet and Let values from the Excel file.
#After merging, the script creates a new column "kunto_luokka" based on the "RMS_yhd" values, which represent the road condition.
#Finally, the merged dataset is saved to a GeoPackage file named "road_condition.gpkg", which can be used in QGIS for visualization.
import geopandas as gpd
import pandas as pd
from pathlib import Path

#------Create an output folder if it doesn't exist to save the results.------

#This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
#Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

print(output_folder)

#----------Load excel and shapefile data (shapefile is from Digiroad)-----------
excel = pd.read_excel(
    r"C:\Users\Laptop\Downloads\Paallystettyjen_teiden_lahtotiedot_ominaisuus_kuntotiedot_100m_L145695.xlsx",
    header=1
)

#----------Shapefile we need is specifically DR_LINKKI.shp, which contains road segments with their attributes.-----------
roads = gpd.read_file(
    r"C:\Users\Laptop\Downloads\KokoSuomi_DIGIROAD_R\PIRKANMAA\PIRKANMAA_2\DR_LINKKI.shp"
)

#-------Filtering by region.-------
#Here we filter the excel data to only include rows where the "Maakunta" column is "Pirkanmaa".
#This was for testing reasons to reduce the dataset size.
#This needs to be modified if we want to have a filter system for the user to select the region of interest.
excel = excel[excel["Maakunta"] == "Pirkanmaa"]

#-------Next, we select only the relevant columns from both datasets and rename them to have consistent naming for merging.-----------
excel = excel[[
    "Tie",
    "Aosa",
    "Aet",
    "Let",
    "RMS_yhd",
    "RIDE_lk"
]]

#-------Aet and Let needs to be renamed to not conflict with the AET and LET columns in the roads dataset.-------
excel = excel.rename(columns={
    "Aet": "Aet_excel",
    "Let": "Let_excel"
})


#-----From the roads dataset, we select the relevant columns.-------
roads = roads[[
    "TIENUMERO",
    "TIEOSANRO",
    "AET",
    "LET",
    "geometry"
]]

#---------We rename the columns in the roads dataset to match the excel dataset for merging.---------
roads = roads.rename(columns={
    "TIENUMERO": "Tie",
    "TIEOSANRO": "Aosa"
})

#--------Merging the two datasets on the "Tie" and "Aosa" columns, which represent the road number and segment number.---------
merged = roads.merge(excel, on=["Tie", "Aosa"])

#------After merging, we filter the merged dataset to only include rows where the AET and LET values
# from the roads dataset are within the Aet_excel and Let_excel values from the excel dataset.-------
merged = merged[
    (merged["AET"] <= merged["Let_excel"]) &
    (merged["LET"] >= merged["Aet_excel"])
]

#------Droping the Aet_excel and Let_excel columns as they are no longer needed after filtering.----------
merged = merged.drop(columns=["Aet_excel", "Let_excel"])

#------This was a proof of concept and will be heavily modified in the future.---------
merged["kunto_luokka"] = pd.cut(
    merged["RMS_yhd"],
    bins=[0,0.3,0.5,0.7,1],
    labels=["Erittäin hyvä", "Hyvä", "Tyydyttävä", "Huono"]
)

#-------Printing the shape and head of the merged dataset to verify the results before plotting and saving to file.---------
print(merged.shape)
print(merged.head())

#------Plotting the merged dataset with the "RMS_yhd" column as the color and a legend to show the different road conditions.---------
merged.plot(column="RMS_yhd", legend=True)


#------Here we define a function to convert the RMS_yhd values to color hex codes for better visualization in QGIS.--------
def rms_to_color(value):
    if pd.isna(value):
        return "#808080"  # Gray for NaN values
    elif value <= 0.3:
        return "#00FF00"  # Green for "Erittäin hyvä"
    elif value <= 0.5:
        return "#FFFF00"  # Yellow for "Hyvä"
    elif value <= 0.7:
        return "#FFA500"  # Orange for "Tyydyttävä"
    else:
        return "#FF0000"  # Red for "Huono"

#Applying the rms_to_color function to the "RMS_yhd" column to create a new "color_hex" column,
# that contains the corresponding color hex codes for each road segment based on its condition.
merged["color_hex"] = merged["RMS_yhd"].apply(rms_to_color)



#Saving the merged dataset to a GeoPackage file named "road_condition.gpkg".
#This is the file that will be used in the QGIS project to visualize the road conditions.
merged.to_file(output_folder / "road_condition.gpkg", driver="GPKG")


#This script is a proof of concept and will be heavily modified in the future to include more features and better visualization options.

#future ideas (feel free to add more):
# - Add filter system for the user to select the region of interest
# - Add dynamic file loading system to allow the user to select their own files for processing
# - Optimize performance.......
# - .......